# Document Processing Pipeline

## 1 · Dependencies

In [1]:
import importlib.util as u, subprocess as sp, sys, os
os.environ["PIP_DISABLE_PIP_VERSION_CHECK"]="1"
def ok(n):
    try: return u.find_spec(n) is not None
    except ModuleNotFoundError: return False
r=[("timm","timm"),("pdf2image","pdf2image"),("pdfplumber","pdfplumber"),("docx","python-docx"),("pptx","python-pptx"),("openpyxl","openpyxl")]
m=[p for n,p in r if not ok(n)]
if m: sp.check_call([sys.executable,"-m","pip","install","-q","--no-deps","--disable-pip-version-check",*m])
x=[n for n,_ in r if not ok(n)]
if x: raise RuntimeError(f"Missing: {x}")
print("Dependencies ready")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 14.9 MB/s eta 0:00:00
Dependencies ready


## 2 · Configuration

In [2]:
import os
from pathlib import Path

B='/kaggle/input/datasets'
CFG={
    'output_dir':   '/kaggle/working/results',
    'input_dir':    '/kaggle/input/test-documents',
    'ds_rvlcdip':   f'{B}/pdavpoojan/the-rvlcdip-dataset-test/test',
    'ds_tobacco':   f'{B}/patrickaudriaz/tobacco3482jpg/Tobacco3482-jpg',
    'ds_isic_imgs': f'{B}/nischaydnk/isic-2019-jpg-224x224-resized/train-image/image',
    'ds_mura':      f'{B}/cjinny/mura-v11/MURA-v1.1',
    'ds_nih':       f'{B}/organizations/nih-chest-xrays/data',
    'ds_disaster':  f'{B}/varpit94/disaster-images-dataset/Comprehensive Disaster Dataset(CDD)',
    'ds_damaged':   f'{B}/mehrael/damaged-buildings-dataset/DamadgedBuildingsDataset',
    'ds_hurricane': f'{B}/kmader/satellite-images-of-hurricane-damage',
    'ds_screenshots':f'{B}/patzold/screenshots-dataset',
    'ds_android':   f'{B}/uzairkhan45/categorized-android-apps-screenshots/Categorized_Andriod_Apps_Images',
    'ds_cedar':     f'{B}/shreelakshmigp/cedardataset/signatures',
    'ds_sigver':    f'{B}/akashgundu/signature-verification-dataset/extract',
    'ds_signet':    f'{B}/victordibia/signverod',
    'ds_ahcd_imgs': f'{B}/skanderkammoun/ahcd-arabic-handwriting/Arabic Handwritten Characters Dataset CSV/training images/training images.csv',
    'ds_indoor':    f'{B}/itsahmad/indoor-scenes-cvpr-2019/indoorCVPR_09/Images',
    'ds_kadid_imgs':f'{B}/srachejack/kadid10k/images',
    'ds_kadid_csv': f'{B}/srachejack/kadid10k/image_labeled_by_per_noise.csv',
    'ds_biq_dir':   f'{B}/nisarahmedrana/biq2021',
    'ds_biq_csv':   f'{B}/nisarahmedrana/biq2021/BIQ2021.csv',
    'ds_casia':     f'{B}/divg07/casia-20-image-tampering-detection-dataset/CASIA2',
    'ds_realfake':  f'{B}/shivamardeshna/real-and-fake-images-dataset-for-image-forensics/Data Set 1/Data Set 1',
    'ds_cg1050_tr': f'{B}/saurabhshahane/cg1050/TRAINING_CG-1050/TRAINING',
    'ds_cg1050_val':f'{B}/saurabhshahane/cg1050/VALIDATION_CG-1050/VALIDATION',
    'img_size':224, 'dpi':300,
    'train_epochs_frozen':4, 'train_epochs_full':10,
    'cnn1_epochs_full':18, 'cnn1_patience':5,
    'cnn2_epochs_full':10, 'cnn2_patience':4,
    'cnn3_epochs_frozen':3, 'cnn3_epochs_full':24, 'cnn3_patience':7,
    'cnn4_epochs_full':14, 'cnn4_patience':5,
    'train_lr_frozen':1.5e-3, 'train_lr_full':2e-4, 'train_lr_backbone':2e-5,
    'train_batch_size':64,
    'train_batch_size_c2':32,
    'mixup_alpha':0.35, 'cutmix_prob':0.25,
    'cnn1_mixup_alpha':0.24, 'cnn1_cutmix_prob':0.08, 'cnn1_hierarchy_weight':0.16,
    'label_smoothing':0.08, 'label_smoothing_cnn4':0.04,
    'cnn3_label_smoothing':0.015, 'cnn3_focal_gamma':0.6, 'cnn3_mixup_alpha':0.0,
    'cnn3_lr_full':1.1e-4, 'cnn3_lr_backbone':8.0e-6, 'cnn3_ordinal_weight':0.20, 'cnn3_soft_smoothing':0.16,
    'focal_gamma':2.0,
    'cnn1_weight_power':0.38, 'cnn3_weight_power':0.24,
    'early_stop_patience':4, 'ema_decay':0.999,
    'ood_max_softmax':0.4, 'ambiguity_gap':0.15, 'tampering_threshold':0.5,
    'decision_tune_models':True, 'decision_threshold_points':181, 'decision_bias_bound':0.28,
    'decision_bias_steps':15, 'decision_bias_rounds':3, 'decision_min_val_gain':0.0002,
    'final_tta':True, 'inference_tta':True,
    'run_captioning':True, 'run_ocr':True,
    'xai_local_examples':6, 'xai_occlusion_grid':4, 'xai_threshold_points':19,
    'xai_lime_examples':2, 'xai_lime_samples':64, 'xai_lime_segments':28,
    'report_latency_reps':12,
    'cnn1_path':'/kaggle/working/cnn1_content.pt',
    'cnn2_path':'/kaggle/working/cnn2_evidence.pt',
    'cnn3_path':'/kaggle/working/cnn3_quality.pt',
    'cnn4_path':'/kaggle/working/cnn4_tamper.pt',
}
Path(CFG['output_dir']).mkdir(parents=True,exist_ok=True)
print('Config loaded')

Config loaded


## 2.5 · Clean Previous Run

In [3]:
import shutil, glob
for p in [CFG['cnn1_path'],CFG['cnn2_path'],CFG['cnn3_path'],CFG['cnn4_path']]:
    if os.path.exists(p): os.remove(p); print(f'Deleted {p}')
for d in ['/kaggle/working/ahcd_imgs', CFG['output_dir']]:
    if os.path.exists(d): shutil.rmtree(d); print(f'Deleted {d}')
for f in glob.glob('/kaggle/working/*.pt')+glob.glob('/kaggle/working/*.json'):
    os.remove(f); print(f'Deleted {f}')
print('Clean')

Deleted /kaggle/working/results
Clean


## 3 · Imports & Utilities

In [4]:
import gc,hashlib,time,warnings,json,glob,logging
from pathlib import Path
from collections import Counter,defaultdict
import numpy as np, pandas as pd, cv2
import torch, torchvision
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader,WeightedRandomSampler
from torchvision import transforms,models
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score,classification_report,roc_auc_score,accuracy_score,confusion_matrix,precision_recall_fscore_support
from sklearn.cluster import KMeans as KM
from PIL import Image
import timm
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.ERROR)
for _n in ['transformers','paddleocr','surya','urllib3','huggingface_hub']:
    logging.getLogger(_n).setLevel(logging.ERROR)

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    _gp=torch.cuda.get_device_properties(0)
    print(f'GPU: {_gp.name} | VRAM: {_gp.total_memory/1e9:.1f} GB')
else: print('WARNING: No GPU')

CONTENT_CLASSES=['injury_photo','scene_damage','screenshot',
                 'typed_document','correspondence',
                 'handwritten','medical_record','data_chart',
                 'signature_stamp','irrelevant']
C1_C2I={c:i for i,c in enumerate(CONTENT_CLASSES)}
C1_I2C={i:c for c,i in C1_C2I.items()}
N_C1=len(CONTENT_CLASSES)

C1_GROUPS=['visual_evidence','document_text','document_mark','irrelevant']
C1_GROUP_MAP={'injury_photo':'visual_evidence','scene_damage':'visual_evidence','screenshot':'visual_evidence','typed_document':'document_text','correspondence':'document_text','medical_record':'document_text','data_chart':'document_text','handwritten':'document_mark','signature_stamp':'document_mark','irrelevant':'irrelevant'}
C1_GROUP_C2I={c:i for i,c in enumerate(C1_GROUPS)}
C1_GROUP_ID=torch.tensor([C1_GROUP_C2I[C1_GROUP_MAP[c]] for c in CONTENT_CLASSES],dtype=torch.long)

EVIDENCE_CLASSES=['primary_evidence','secondary_evidence']
C2_C2I={c:i for i,c in enumerate(EVIDENCE_CLASSES)}
C2_I2C={i:c for c,i in C2_C2I.items()}
N_C2=len(EVIDENCE_CLASSES)

QUALITY_CLASSES=['clear','acceptable','degraded']
C3_C2I={c:i for i,c in enumerate(QUALITY_CLASSES)}
C3_I2C={i:c for c,i in C3_C2I.items()}
N_C3=len(QUALITY_CLASSES)

TAMPER_CLASSES=['authentic','tampered']
TAMP_C2I={c:i for i,c in enumerate(TAMPER_CLASSES)}
TAMP_I2C={i:c for c,i in TAMP_C2I.items()}
N_C4=len(TAMPER_CLASSES)

print(f'CNN1({N_C1}) CNN2({N_C2}) CNN3({N_C3}) CNN4({N_C4})')

def free_vram():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for c in iter(lambda:f.read(65536),b''): h.update(c)
    return h.hexdigest()[:16]

def find_imgs(root,exts=('.jpg','.jpeg','.png','.bmp','.tif','.tiff')):
    fs=[]
    for e in exts: fs.extend(Path(root).rglob(f'*{e}'))
    return sorted(set(fs))

GPU: Tesla T4 | VRAM: 15.6 GB
CNN1(10) CNN2(2) CNN3(3) CNN4(2)


## 4 · Stage 0 — File Normalisation

In [5]:
def normalize_file(filepath):
    fp=Path(filepath); ext=fp.suffix.lower(); pages=[]
    if ext=='.pdf':
        from pdf2image import convert_from_path
        pages=convert_from_path(str(fp),dpi=CFG['dpi'])
    elif ext in ('.doc','.docx','.pptx','.ppt'):
        import subprocess
        tmp=Path('/tmp/lo_out'); tmp.mkdir(exist_ok=True)
        subprocess.run(['libreoffice','--headless','--convert-to','pdf',
                        '--outdir',str(tmp),str(fp)],capture_output=True,timeout=60)
        pp=tmp/f'{fp.stem}.pdf'
        if pp.exists():
            from pdf2image import convert_from_path
            pages=convert_from_path(str(pp),dpi=CFG['dpi'])
    elif ext in ('.jpg','.jpeg','.png','.bmp','.tif','.tiff','.webp'):
        pages=[Image.open(str(fp)).convert('RGB')]
    return pages

def extract_native_text(filepath):
    if Path(filepath).suffix.lower()!='.pdf': return {}
    try:
        import pdfplumber
        out={}
        with pdfplumber.open(filepath) as pdf:
            for i,pg in enumerate(pdf.pages):
                t=pg.extract_text()
                if t and len(t.strip())>10: out[i]=t.strip()
        return out
    except: return {}

print('Stage 0 ready')

Stage 0 ready


## 5 · Dataset Scanners

In [6]:
RVL_MAP={
    'advertisement':'irrelevant',   'budget':'data_chart',
    'email':'correspondence',       'file_folder':'irrelevant',
    'form':'typed_document',        'handwritten':'handwritten',
    'invoice':'typed_document',     'letter':'correspondence',
    'memo':'correspondence',        'news_article':'typed_document',
    'presentation':'data_chart',    'questionnaire':'typed_document',
    'resume':'typed_document',      'scientific_publication':'typed_document',
    'scientific_report':'typed_document', 'specification':'typed_document',
}
def scan_rvlcdip():
    root=CFG['ds_rvlcdip']; s=[]
    if not os.path.isdir(root): return s
    for d in sorted(Path(root).iterdir()):
        if not d.is_dir(): continue
        l=RVL_MAP.get(d.name)
        if l: [s.append((str(f),l,'rvlcdip')) for f in find_imgs(d)]
    return s

TOBACCO_MAP={
    'ADVE':'irrelevant',    'Email':'correspondence',
    'Form':'typed_document','Letter':'correspondence',
    'Memo':'correspondence','News':'typed_document',
    'Note':'handwritten',   'Report':'typed_document',
    'Resume':'typed_document','Scientific':'typed_document',
}
def scan_tobacco():
    root=CFG['ds_tobacco']; s=[]
    if not os.path.isdir(root): return s
    for d in sorted(Path(root).iterdir()):
        if not d.is_dir(): continue
        l=TOBACCO_MAP.get(d.name)
        if l: [s.append((str(f),l,'tobacco')) for f in find_imgs(d)]
    return s

def scan_isic():
    r=CFG['ds_isic_imgs']
    return [(str(f),'injury_photo','isic') for f in find_imgs(r)] if os.path.isdir(r) else []

def scan_mura():
    r=CFG['ds_mura']; s=[]
    for sp in ('train','valid'):
        p=os.path.join(r,sp)
        if os.path.isdir(p): s.extend((str(f),'medical_record','mura') for f in find_imgs(p))
    return s

def scan_nih():
    r=CFG['ds_nih']; s=[]
    if not os.path.isdir(r): return s
    for d in sorted(Path(r).glob('images_*/images')):
        for f in list(find_imgs(d))[:5000]: s.append((str(f),'medical_record','nih'))
        if len(s)>=15000: break
    return s

DISASTER_DMG={'Damaged_Infrastructure','Fire_Disaster','Human_Damage','Land_Disaster','Water_Disaster'}
def scan_disaster():
    r=CFG['ds_disaster']; s=[]
    if not os.path.isdir(r): return s
    for d in sorted(Path(r).iterdir()):
        if not d.is_dir(): continue
        if d.name in DISASTER_DMG:
            s.extend((str(f),'scene_damage','disaster') for f in find_imgs(d))
        elif d.name=='Non_Damage':
            s.extend((str(f),'irrelevant','disaster') for f in find_imgs(d))
    return s

def scan_damaged():
    r=CFG['ds_damaged']; s=[]
    if not os.path.isdir(r): return s
    for d in ('db','db2','Aug_db'):
        p=os.path.join(r,d)
        if os.path.isdir(p): s.extend((str(f),'scene_damage','damaged') for f in find_imgs(p))
    return s

def scan_hurricane():
    r=CFG['ds_hurricane']; s=[]
    if not os.path.isdir(r): return s
    for sd in Path(r).iterdir():
        if not sd.is_dir(): continue
        if (sd/'damage').is_dir():
            s.extend((str(f),'scene_damage','hurricane') for f in find_imgs(sd/'damage'))
        if (sd/'no_damage').is_dir():
            s.extend((str(f),'irrelevant','hurricane') for f in find_imgs(sd/'no_damage'))
    return s

def scan_screenshots():
    r=CFG['ds_screenshots']
    return [(str(f),'screenshot','screenshots') for f in find_imgs(r)] if os.path.isdir(r) else []

def scan_android():
    r=CFG['ds_android']
    return [(str(f),'screenshot','android') for f in find_imgs(r)] if os.path.isdir(r) else []

def scan_cedar():
    r=CFG['ds_cedar']; s=[]
    for sub in ('full_org','full_forg'):
        p=os.path.join(r,sub)
        if os.path.isdir(p): s.extend((str(f),'signature_stamp','cedar') for f in find_imgs(p))
    return s

def scan_sigver():
    r=CFG['ds_sigver']
    return [(str(f),'signature_stamp','sigver') for f in find_imgs(r)] if os.path.isdir(r) else []

def scan_signet():
    r=CFG['ds_signet']
    if not os.path.isdir(r): return []
    return [(str(f),'signature_stamp','signet') for f in find_imgs(r)]

_AHCD_CACHE=Path('/kaggle/working/ahcd_imgs')
def scan_ahcd():
    existing=list(_AHCD_CACHE.glob('*.png')) if _AHCD_CACHE.exists() else []
    if len(existing)>=100:
        return [(str(f),'handwritten','ahcd') for f in sorted(existing)]
    p=CFG['ds_ahcd_imgs']
    if not os.path.exists(p): return []
    df=pd.read_csv(p,header=None,nrows=5000)
    _AHCD_CACHE.mkdir(exist_ok=True); s=[]
    for i in range(min(len(df),3000)):
        px=df.iloc[i].values.astype(np.uint8)
        if len(px)<4096: continue
        img=Image.fromarray(px[:4096].reshape(64,64),mode='L').convert('RGB')
        fp=_AHCD_CACHE/f'ahcd_{i}.png'; img.save(fp)
        s.append((str(fp),'handwritten','ahcd'))
    return s

def scan_indoor():
    r=CFG['ds_indoor']
    return [(str(f),'irrelevant','indoor') for f in find_imgs(r)] if os.path.isdir(r) else []

def scan_kadid():
    cp=CFG['ds_kadid_csv']; idir=CFG['ds_kadid_imgs']; s=[]
    if not os.path.exists(cp): return s
    df=pd.read_csv(cp)
    for _,row in df.iterrows():
        fp=os.path.join(idir,str(row['image']))
        if not os.path.exists(fp): continue
        d=float(row['dmos'])
        l='clear' if d>=4.0 else 'acceptable' if d>=2.5 else 'degraded'
        s.append((fp,l,'kadid'))
    return s

def scan_biq2021():
    cp=CFG['ds_biq_csv']; bd=CFG['ds_biq_dir']; s=[]
    if not os.path.exists(cp): return s
    df=pd.read_csv(cp)
    for _,row in df.iterrows():
        fp=os.path.join(bd,str(row['Images']))
        if not os.path.exists(fp): continue
        m=float(row['MOS'])
        l='clear' if m>=0.65 else 'acceptable' if m>=0.40 else 'degraded'
        s.append((fp,l,'biq2021'))
    return s

def scan_casia():
    r=CFG['ds_casia']; s=[]
    au,tp=os.path.join(r,'Au'),os.path.join(r,'Tp')
    if os.path.isdir(au): s.extend((str(f),'authentic','casia') for f in find_imgs(au))
    if os.path.isdir(tp): s.extend((str(f),'tampered','casia') for f in find_imgs(tp))
    return s

def scan_realfake():
    r=CFG['ds_realfake']; s=[]
    if not os.path.isdir(r): return s
    for sp in ('train','test','validation'):
        for ld,lb in [('fake','tampered'),('real','authentic')]:
            p=os.path.join(r,sp,ld)
            if os.path.isdir(p): s.extend((str(f),lb,'realfake') for f in find_imgs(p))
    return s

def scan_cg1050():
    s=[]
    for base in (CFG['ds_cg1050_tr'],CFG['ds_cg1050_val']):
        for sub,lb in [('ORIGINAL','authentic'),('TAMPERED','tampered')]:
            p=os.path.join(base,sub)
            if os.path.isdir(p): s.extend((str(f),lb,'cg1050') for f in find_imgs(p))
    return s

print('Scanners ready')

Scanners ready


## 6 · Data Processing Utilities

In [7]:
def deduplicate(samples):
    seen=set(); out=[]
    for s in samples:
        h=sha256_file(s[0])
        if h not in seen: seen.add(h); out.append(s)
    if len(samples)!=len(out): print(f'  Dedup: {len(samples)} -> {len(out)}')
    return out

def compute_weights(samples,c2i,power=0.5,scale=None):
    counts=Counter(s[1] for s in samples); total=len(samples); scale=scale or {}
    w=torch.zeros(len(c2i))
    for cls,idx in c2i.items():
        c=counts.get(cls,1); w[idx]=(total/(len(c2i)*c))**power*scale.get(cls,1.0)
    return w

def prepare_splits(samples,test_size=0.15,val_size=0.15):
    labels=[s[1] for s in samples]
    sc=Counter(labels)
    if min(sc.values())<2: raise ValueError(f'Class with <2 samples: {sc}')
    trv,test=train_test_split(samples,test_size=test_size,stratify=labels,random_state=42)
    trv_l=[s[1] for s in trv]; vr=val_size/(1-test_size)
    tr,val=train_test_split(trv,test_size=vr,stratify=trv_l,random_state=42)
    print(f'  Split: {len(tr)} train / {len(val)} val / {len(test)} test')
    return tr,val,test

sz=CFG['img_size']

train_tf=transforms.Compose([
    transforms.Resize((sz+32,sz+32)),
    transforms.RandomCrop(sz),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomPerspective(distortion_scale=0.2,p=0.3),
    transforms.ColorJitter(0.3,0.3,0.2,0.08),
    transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225]),
    transforms.RandomErasing(p=0.2,scale=(0.02,0.15)),
])

train_tf_q3=transforms.Compose([
    transforms.Resize((sz,sz)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225]),
])

train_tf_c4=transforms.Compose([
    transforms.Resize((sz,sz)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.ColorJitter(0.1,0.1,0.05,0.02),
    transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225]),
])

val_tf=transforms.Compose([
    transforms.Resize((sz,sz)),
    transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225]),
])

print('Data utils ready')

Data utils ready


## 7 · Model Definitions

In [8]:
class ImgDS(Dataset):
    def __init__(self,samples,c2i,tf=None):
        self.samples=samples; self.c2i=c2i; self.tf=tf
    def __len__(self): return len(self.samples)
    def __getitem__(self,i):
        p,l,_=self.samples[i]
        try: img=Image.open(p).convert('RGB')
        except: img=Image.new('RGB',(sz,sz))
        if self.tf: img=self.tf(img)
        return img,self.c2i[l]

class FocalLoss(nn.Module):
    def __init__(self,gamma=2.0,weight=None,ls=0.0):
        super().__init__(); self.gamma=gamma; self.w=weight; self.ls=ls
    def forward(self,x,y):
        ce=F.cross_entropy(x,y,weight=self.w,label_smoothing=self.ls,reduction='none')
        return ((1-torch.exp(-ce))**self.gamma*ce).mean()

class HierarchicalFocalLoss(nn.Module):
    def __init__(self,gamma=2.0,weight=None,ls=0.0,group_ids=None,gw=0.15):
        super().__init__(); self.base=FocalLoss(gamma,weight,ls); self.group_ids=group_ids; self.gw=gw
    def forward(self,x,y):
        loss=self.base(x,y)
        if self.group_ids is None or self.gw<=0: return loss
        gids=self.group_ids.to(x.device); target=gids[y]; vals=torch.unique(gids).sort()[0]
        glogits=torch.stack([torch.logsumexp(x[:,gids==g],dim=1) for g in vals],dim=1)
        return loss+self.gw*F.cross_entropy(glogits,target)

class SoftOrdinalQualityLoss(nn.Module):
    def __init__(self,weight=None,ls=0.015,ow=0.20,smooth=0.16):
        super().__init__(); self.w=weight; self.ls=ls; self.ow=ow; self.smooth=smooth
    def forward(self,x,y):
        n=x.size(1); target=torch.zeros_like(x); main=1.0-self.smooth
        target.scatter_(1,y[:,None],main)
        left=(y-1).clamp_min(0); right=(y+1).clamp_max(n-1)
        for side in [left,right]:
            mask=side!=y
            if mask.any(): target[mask,side[mask]]+=self.smooth/2
        rowsum=target.sum(1,keepdim=True).clamp_min(1e-6); target=target/rowsum
        logp=F.log_softmax(x,dim=1); ce=-(target*logp).sum(1)
        if self.w is not None:
            ww=(target*self.w.to(x.device)[None,:]).sum(1)/self.w.to(x.device).mean().clamp_min(1e-6); ce=ce*ww
        p=F.softmax(x,dim=1); idx=torch.arange(n,device=x.device,dtype=p.dtype)
        reg=F.smooth_l1_loss((p*idx).sum(1),y.float(),reduction='none')
        return ce.mean()+self.ow*reg.mean()

class OrdinalQualityLoss(nn.Module):
    def __init__(self,weight=None,ls=0.0,ow=0.25):
        super().__init__(); self.w=weight; self.ls=ls; self.ow=ow
    def forward(self,x,y):
        ce=F.cross_entropy(x,y,weight=self.w,label_smoothing=self.ls)
        p=F.softmax(x,dim=1); idx=torch.arange(x.size(1),device=x.device,dtype=p.dtype)
        reg=F.smooth_l1_loss((p*idx).sum(1),y.float(),reduction='none')
        if self.w is not None: reg=reg*self.w[y]/self.w.mean().clamp_min(1e-6)
        return ce+self.ow*reg.mean()

def mixup(x,y,a=0.2):
    lam=np.random.beta(a,a) if a>0 else 1.0
    idx=torch.randperm(x.size(0)).to(x.device)
    return lam*x+(1-lam)*x[idx],y,y[idx],lam

def cutmix(x,y):
    lam=np.random.beta(1,1); B,C,H,W=x.shape
    idx=torch.randperm(B).to(x.device)
    cx,cy=np.random.randint(W),np.random.randint(H)
    rw,rh=int(W*np.sqrt(1-lam))//2,int(H*np.sqrt(1-lam))//2
    x1,y1,x2,y2=max(cx-rw,0),max(cy-rh,0),min(cx+rw,W),min(cy+rh,H)
    xc=x.clone(); xc[:,:,y1:y2,x1:x2]=x[idx,:,y1:y2,x1:x2]
    return xc,y,y[idx],1-(x2-x1)*(y2-y1)/(W*H)

def mix_loss(crit,pred,ya,yb,lam):
    return lam*crit(pred,ya)+(1-lam)*crit(pred,yb)

def freeze_batchnorm(m):
    for x in m.modules():
        if isinstance(x,(nn.BatchNorm1d,nn.BatchNorm2d,nn.BatchNorm3d)):
            x.eval()
            for p in x.parameters(): p.requires_grad=False

class EMA:
    def __init__(self,model,decay=0.999):
        self.shadow={k:v.clone().detach() for k,v in model.state_dict().items()}
        self.d=decay
    def update(self,model):
        for k,v in model.state_dict().items():
            if torch.is_floating_point(v): self.shadow[k]=self.d*self.shadow[k]+(1-self.d)*v.detach()
            else: self.shadow[k]=v.clone().detach()
    def apply(self,model): model.load_state_dict(self.shadow)

class GeM(nn.Module):
    def __init__(self,p=3,eps=1e-6):
        super().__init__()
        self.p=nn.Parameter(torch.ones(1)*p); self.eps=eps
    def forward(self,x):
        return F.adaptive_avg_pool2d(x.clamp(min=self.eps).pow(self.p),(1,1)).pow(1./self.p)

class CNN1(nn.Module):
    def __init__(self,nc=N_C1):
        super().__init__()
        self.backbone=timm.create_model('tf_efficientnetv2_s',pretrained=True,num_classes=0,global_pool='')
        feat_dim=self.backbone.num_features
        self.pool=GeM()
        self.head=nn.Sequential(
            nn.Flatten(),nn.Dropout(0.4),
            nn.Linear(feat_dim,512),nn.ReLU(),nn.BatchNorm1d(512),
            nn.Dropout(0.3),nn.Linear(512,nc))
        self._embed_dim=feat_dim
    def forward(self,x): return self.head(self.pool(self.backbone(x)))
    def embed(self,x): return self.pool(self.backbone(x)).flatten(1)
    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad=False
    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad=True

class CNN2(nn.Module):
    def __init__(self,embed_dim=1280,nc=N_C2):
        super().__init__()
        bb=models.resnext50_32x4d(weights=models.ResNeXt50_32X4D_Weights.DEFAULT)
        self.backbone=nn.Sequential(*list(bb.children())[:-1])
        self.head=nn.Sequential(
            nn.Flatten(),nn.Dropout(0.5),
            nn.Linear(2048+embed_dim,512),nn.ReLU(),nn.BatchNorm1d(512),
            nn.Dropout(0.3),nn.Linear(512,nc))
    def forward(self,x,c1e):
        return self.head(torch.cat([self.backbone(x).flatten(1),c1e],dim=1))
    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad=False
    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad=True

class CNN3(nn.Module):
    def __init__(self,nc=N_C3):
        super().__init__()
        self.backbone=timm.create_model('tf_efficientnet_b0',pretrained=True,num_classes=0,global_pool='avg')
        feat_dim=self.backbone.num_features
        self.head=nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(feat_dim,128),nn.ReLU(),nn.BatchNorm1d(128),
            nn.Dropout(0.2),nn.Linear(128,nc))
    def forward(self,x): return self.head(self.backbone(x))
    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad=False
    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad=True

class CNN4(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone=timm.create_model('tf_efficientnet_b1',pretrained=True,num_classes=0,global_pool='avg')
        feat_dim=self.backbone.num_features
        self.head=nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(feat_dim,128),nn.ReLU(),nn.BatchNorm1d(128),
            nn.Dropout(0.2),nn.Linear(128,2))
    def forward(self,x): return self.head(self.backbone(x))
    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad=False
    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad=True

print('Models defined')
for nm,M in [('CNN1 EfficientNetV2-S+GeM',CNN1),('CNN2 ResNeXt-50',lambda:CNN2()),
             ('CNN3 EfficientNet-B0',CNN3),('CNN4 EfficientNet-B1',CNN4)]:
    m=M() if callable(M) else M
    print(f'  {nm}: {sum(p.numel() for p in m.parameters()):,} params')
    del m

Models defined


model.safetensors:   0%|          | 0.00/86.5M [00:00<?, ?B/s]

  CNN1 EfficientNetV2-S+GeM: 20,839,515 params
Downloading: "https://download.pytorch.org/models/resnext50_32x4d-1a0047aa.pth" to /root/.cache/torch/hub/checkpoints/resnext50_32x4d-1a0047aa.pth


100%|██████████| 95.8M/95.8M [00:00<00:00, 157MB/s]


  CNN2 ResNeXt-50: 24,686,402 params


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

  CNN3 EfficientNet-B0: 4,172,159 params


model.safetensors:   0%|          | 0.00/31.5M [00:00<?, ?B/s]

  CNN4 EfficientNet-B1: 6,677,666 params


## 8 · Evaluation & Calibration

In [9]:
def forward_logits(model,imgs,c1m=None):
    if c1m is not None:
        with torch.inference_mode(): e=c1m.embed(imgs)
        return model(imgs,e)
    return model(imgs)

@torch.no_grad()
def collect_logits(model,loader,c1m=None,tta=False):
    model.eval(); lo,la=[],[]
    for imgs,labels in loader:
        imgs,labels=imgs.to(DEVICE),labels.to(DEVICE)
        out=forward_logits(model,imgs,c1m)
        if tta: out=(out+forward_logits(model,torch.flip(imgs,dims=[3]),c1m))/2
        lo.append(out.detach()); la.append(labels.detach())
    return torch.cat(lo),torch.cat(la)

def decision_logits(logits,T=1.0,decision=None):
    z=logits/max(float(T),1e-6)
    if decision and decision.get('type')=='logit_bias':
        b=torch.tensor(decision.get('bias',[]),device=z.device,dtype=z.dtype)
        if b.numel()==z.shape[1]: z=z+b.view(1,-1)
    return z

def decision_probs(logits,T=1.0,decision=None):
    return F.softmax(decision_logits(logits,T,decision),dim=1)

def decision_preds(probs,decision=None):
    if decision and decision.get('type')=='binary_threshold' and probs.shape[1]==2:
        return (probs[:,1]>=float(decision.get('threshold',0.5))).astype(int)
    return probs.argmax(1)

@torch.no_grad()
def evaluate(model,loader,c1m=None,T=1.0,tta=False,decision=None):
    lo,la=collect_logits(model,loader,c1m,tta)
    pr=decision_probs(lo,T,decision).cpu().numpy()
    ps=decision_preds(pr,decision).tolist(); ls=la.cpu().tolist(); labs=list(range(pr.shape[1]))
    mf1=f1_score(ls,ps,average='macro',labels=labs,zero_division=0)
    pf1=f1_score(ls,ps,average=None,labels=labs,zero_division=0)
    return mf1,pf1,ps,ls

def _softmax_np(x):
    x=x-x.max(axis=1,keepdims=True); e=np.exp(x); return e/np.clip(e.sum(axis=1,keepdims=True),1e-12,None)

def tune_binary_threshold_from_logits(lo,la,T,class_names):
    z=(lo/max(float(T),1e-6)).detach().cpu().numpy(); labels=la.cpu().numpy(); probs=_softmax_np(z); labs=list(range(probs.shape[1]))
    base_preds=probs.argmax(1); base=f1_score(labels,base_preds,average='macro',labels=labs,zero_division=0)
    best_t,best=0.5,base
    for t in np.linspace(0.05,0.95,CFG.get('decision_threshold_points',181)):
        preds=(probs[:,1]>=t).astype(int); val=f1_score(labels,preds,average='macro',labels=labs,zero_division=0)
        if val>best: best_t,best=float(t),float(val)
    if best-base<CFG.get('decision_min_val_gain',0.0002): return {'type':'argmax','base_val_f1':round(float(base),6),'decision_val_f1':round(float(base),6),'class_names':class_names}
    return {'type':'binary_threshold','threshold':round(best_t,5),'base_val_f1':round(float(base),6),'decision_val_f1':round(float(best),6),'class_names':class_names}

def tune_logit_bias_from_logits(lo,la,T,class_names):
    z=(lo/max(float(T),1e-6)).detach().cpu().numpy(); labels=la.cpu().numpy(); n=z.shape[1]; labs=list(range(n))
    base_probs=_softmax_np(z); base=f1_score(labels,base_probs.argmax(1),average='macro',labels=labs,zero_division=0)
    bias=np.zeros(n,dtype=np.float32); best=float(base)
    vals=np.linspace(-CFG.get('decision_bias_bound',0.28),CFG.get('decision_bias_bound',0.28),CFG.get('decision_bias_steps',15))
    for _ in range(CFG.get('decision_bias_rounds',3)):
        moved=False
        for c in range(n):
            cur=bias[c]; local_v,local=cur,best
            for v in vals:
                tb=bias.copy(); tb[c]=v
                preds=_softmax_np(z+tb).argmax(1)
                val=f1_score(labels,preds,average='macro',labels=labs,zero_division=0)
                if val>local: local_v,local=float(v),float(val)
            if local>best:
                bias[c]=local_v; best=local; moved=True
        if not moved: break
    if best-base<CFG.get('decision_min_val_gain',0.0002): return {'type':'argmax','base_val_f1':round(float(base),6),'decision_val_f1':round(float(base),6),'class_names':class_names}
    return {'type':'logit_bias','bias':[round(float(x),5) for x in bias.tolist()],'base_val_f1':round(float(base),6),'decision_val_f1':round(float(best),6),'class_names':class_names}

def tune_decision(model,loader,class_names,c1m=None,T=1.0,tta=False):
    if not CFG.get('decision_tune_models',True): return {'type':'argmax','class_names':class_names}
    lo,la=collect_logits(model,loader,c1m,tta)
    if len(class_names)==2: dec=tune_binary_threshold_from_logits(lo,la,T,class_names)
    else: dec=tune_logit_bias_from_logits(lo,la,T,class_names)
    print(f"  Decision={dec.get('type')} val_f1 {dec.get('base_val_f1')} -> {dec.get('decision_val_f1')}")
    return dec

def calibrate_T(model,loader,c1m=None,tta=False):
    lo,la=collect_logits(model,loader,c1m,tta)
    T=nn.Parameter(torch.ones(1,device=DEVICE)*1.5)
    opt=torch.optim.Adam([T],lr=0.01)
    for _ in range(200):
        opt.zero_grad(); loss=F.cross_entropy(lo/T.clamp(0.1,5.0),la); loss.backward(); opt.step()
    T_val=T.clamp(0.1,5.0).item(); print(f'  T={T_val:.3f}')
    return T_val

def ece_score(probs,labels,preds=None,bins=10):
    pred=probs.argmax(1) if preds is None else np.asarray(preds); labels=np.asarray(labels); conf=probs[np.arange(len(probs)),pred]; ece=0.0
    for lo,hi in zip(np.linspace(0,1,bins+1)[:-1],np.linspace(0,1,bins+1)[1:]):
        m=(conf>=lo)&(conf<(hi if hi<1 else hi+1e-9))
        if m.any(): ece+=m.mean()*abs((pred[m]==labels[m]).mean()-conf[m].mean())
    return float(ece)

def auc_metric(probs,labels):
    try:
        labels=np.asarray(labels); n=probs.shape[1]
        if n==2: return float(roc_auc_score(labels,probs[:,1]))
        y=np.eye(n)[labels]
        return float(roc_auc_score(y,probs,average='macro',multi_class='ovr'))
    except Exception: return None

def top_confusions(cm,class_names,k=10):
    out=[]
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            if i!=j and cm[i,j]>0: out.append({'true':class_names[i],'predicted':class_names[j],'count':int(cm[i,j])})
    return sorted(out,key=lambda x:x['count'],reverse=True)[:k]

def save_learning_curve(history,save_path):
    root=Path(CFG['output_dir'])/'training'; root.mkdir(parents=True,exist_ok=True)
    stem=Path(save_path).stem
    with open(root/f'{stem}_history.json','w') as f: json.dump(history,f,indent=2)
    if not history: return
    fig,ax=plt.subplots(1,3,figsize=(14,4))
    xs=list(range(1,len(history)+1))
    ax[0].plot(xs,[h['loss'] for h in history],marker='o'); ax[0].set_title('loss')
    ax[1].plot(xs,[h['acc'] for h in history],marker='o'); ax[1].set_title('train accuracy')
    ax[2].plot(xs,[h['val_f1'] for h in history],marker='o'); ax[2].set_title('val macro F1')
    for a in ax: a.set_xlabel('epoch step'); a.grid(alpha=.25)
    fig.tight_layout(); fig.savefig(root/f'{stem}_learning_curve.png',dpi=160); plt.close(fig)

def last_conv(m):
    found=None
    for _,x in m.named_modules():
        if isinstance(x,nn.Conv2d): found=x
    return found

def grad_cam(model,img,target,c1m=None):
    conv=last_conv(model.backbone if hasattr(model,'backbone') else model)
    if conv is None: return None
    acts=[]; grads=[]
    def fh(_,__,o):
        if torch.is_tensor(o): acts.append(o)
    def bh(_,gi,go):
        if go and torch.is_tensor(go[0]): grads.append(go[0])
    h1=conv.register_forward_hook(fh); h2=conv.register_full_backward_hook(bh)
    try:
        model.zero_grad(set_to_none=True)
        if c1m is not None:
            with torch.inference_mode(): e=c1m.embed(img)
            out=model(img,e)
        else: out=model(img)
        out[0,target].backward()
        if not acts or not grads: return None
        a,g=acts[-1].detach(),grads[-1].detach()
        w=g.mean(dim=(2,3),keepdim=True)
        cam=torch.relu((w*a).sum(1,keepdim=True))
        cam=F.interpolate(cam,img.shape[-2:],mode='bilinear',align_corners=False).squeeze().cpu().numpy()
        return (cam-cam.min())/(cam.max()-cam.min()+1e-9)
    finally:
        h1.remove(); h2.remove()

@torch.no_grad()
def occlusion_drops(model,img,target,c1m=None,T=1.0,grid=4):
    base=F.softmax(forward_logits(model,img,c1m)/T,dim=1)[0,target].item()
    _,_,h,w=img.shape; rows=[]
    for gy in range(grid):
        for gx in range(grid):
            z=img.clone(); y0,y1=gy*h//grid,(gy+1)*h//grid; x0,x1=gx*w//grid,(gx+1)*w//grid; z[:,:,y0:y1,x0:x1]=0
            p=F.softmax(forward_logits(model,z,c1m)/T,dim=1)[0,target].item()
            rows.append({'cell':[gy,gx],'drop':round(base-p,5),'after':round(p,5)})
    return sorted(rows,key=lambda x:x['drop'],reverse=True)

def denorm_img(t):
    im=t.detach().cpu().permute(1,2,0).numpy()
    return np.clip(im*np.array([.229,.224,.225])+np.array([.485,.456,.406]),0,1)

def make_segments(im,n=28):
    try:
        from skimage.segmentation import slic
        seg=slic(im,n_segments=n,compactness=12,start_label=0,channel_axis=-1)
        return seg.astype(np.int32)
    except Exception:
        g=max(2,int(np.sqrt(n))); h,w=im.shape[:2]; seg=np.zeros((h,w),dtype=np.int32); c=0
        for y in range(g):
            for x in range(g):
                seg[y*h//g:(y+1)*h//g,x*w//g:(x+1)*w//g]=c; c+=1
        return seg

@torch.no_grad()
def lime_explain(model,img,target,c1m=None,T=1.0,n_segments=28,n_samples=64):
    im=denorm_img(img[0]); seg=make_segments(im,n_segments); ids=np.unique(seg); m=len(ids)
    if m<2: return None,None
    rng=np.random.default_rng(42+int(target)); masks=[np.ones(m,dtype=np.float32)]
    for _ in range(max(4,n_samples-1)):
        keep=rng.random(m)>.45
        if not keep.any(): keep[rng.integers(0,m)]=True
        masks.append(keep.astype(np.float32))
    ys=[]; bs=16; base=img.detach()
    for st in range(0,len(masks),bs):
        batch=[]
        for mask in masks[st:st+bs]:
            keep_ids=ids[mask.astype(bool)]
            mt=torch.tensor(np.isin(seg,keep_ids),device=base.device,dtype=base.dtype)[None,None]
            batch.append(base*mt)
        xb=torch.cat(batch,0)
        ys.extend(F.softmax(forward_logits(model,xb,c1m)/max(float(T),1e-6),dim=1)[:,target].detach().cpu().numpy().tolist())
    X=np.asarray(masks,dtype=np.float32); y=np.asarray(ys,dtype=np.float32); reg=1e-3*np.eye(m,dtype=np.float32)
    try: coef=np.linalg.solve(X.T@X+reg,X.T@y)
    except Exception: coef=np.linalg.lstsq(X,y,rcond=None)[0]
    heat=np.zeros_like(seg,dtype=np.float32)
    for sid,w in zip(ids,coef): heat[seg==sid]=w
    heat=(heat-heat.min())/(heat.max()-heat.min()+1e-9)
    top=np.argsort(coef)[::-1][:8]
    rows=[{'segment':int(ids[i]),'weight':round(float(coef[i]),6)} for i in top]
    return heat,rows

def save_xai_artifacts(model,loader,class_names,save_path,c1m=None,T=1.0,tta=False,decision=None):
    root=Path(CFG['output_dir'])/'xai'/Path(save_path).stem; root.mkdir(parents=True,exist_ok=True)
    lo,la=collect_logits(model,loader,c1m,tta); probs=decision_probs(lo,T,decision).cpu().numpy(); labels=la.cpu().numpy(); preds=decision_preds(probs,decision)
    labs=list(range(len(class_names))); cm=confusion_matrix(labels,preds,labels=labs)
    acc=accuracy_score(labels,preds); mf1=f1_score(labels,preds,average='macro',labels=labs,zero_division=0); pf1=f1_score(labels,preds,average=None,labels=labs,zero_division=0)
    prec,rec,_,_=precision_recall_fscore_support(labels,preds,labels=labs,zero_division=0); auc=auc_metric(probs,labels)
    report={'accuracy':round(float(acc),5),'macro_f1':round(float(mf1),5),'auc':None if auc is None else round(float(auc),5),'ece':round(ece_score(probs,labels,preds),5),'top_confusions':top_confusions(cm,class_names),'per_class':{class_names[i]:{'precision':round(float(prec[i]),5),'recall':round(float(rec[i]),5),'f1':round(float(pf1[i]),5),'support':int((labels==i).sum())} for i in range(len(class_names))}}
    with open(root/'metrics.json','w') as f: json.dump(report,f,indent=2)
    with open(root/'decision_config.json','w') as f: json.dump(decision or {'type':'argmax','class_names':class_names},f,indent=2)
    with open(root/'classification_report.txt','w') as f: f.write(classification_report(labels,preds,labels=labs,target_names=class_names,zero_division=0))
    fig,ax=plt.subplots(figsize=(max(6,len(class_names)*0.8),max(5,len(class_names)*0.6))); ax.imshow(cm,cmap='Blues')
    ax.set_xticks(range(len(class_names))); ax.set_yticks(range(len(class_names))); ax.set_xticklabels(class_names,rotation=45,ha='right'); ax.set_yticklabels(class_names)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]): ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=8)
    ax.set_xlabel('predicted'); ax.set_ylabel('true'); fig.tight_layout(); fig.savefig(root/'confusion_matrix.png',dpi=160); plt.close(fig)
    conf=probs[np.arange(len(probs)),preds]; ok=(preds==labels).astype(float); xs=[]; ys=[]
    for lo_b,hi_b in zip(np.linspace(0,1,11)[:-1],np.linspace(0,1,11)[1:]):
        m=(conf>=lo_b)&(conf<(hi_b if hi_b<1 else hi_b+1e-9))
        if m.any(): xs.append(float(conf[m].mean())); ys.append(float(ok[m].mean()))
    fig,ax=plt.subplots(figsize=(5,5)); ax.plot([0,1],[0,1],'--',c='black',alpha=.5); ax.plot(xs,ys,marker='o'); ax.set_xlabel('confidence'); ax.set_ylabel('accuracy'); ax.set_title('calibration'); fig.tight_layout(); fig.savefig(root/'calibration.png',dpi=160); plt.close(fig)
    thresholds=[]
    for i,c in enumerate(class_names):
        y=(labels==i).astype(int)
        if y.sum()==0: continue
        best={'class':c,'threshold':0.5,'f1':0.0,'precision':0.0,'recall':0.0}
        for t in np.linspace(0.05,0.95,CFG.get('xai_threshold_points',19)):
            pp=(probs[:,i]>=t).astype(int); tp=((pp==1)&(y==1)).sum(); fp=((pp==1)&(y==0)).sum(); fn=((pp==0)&(y==1)).sum()
            pr=tp/max(1,tp+fp); rc=tp/max(1,tp+fn); f1=0 if pr+rc==0 else 2*pr*rc/(pr+rc)
            if f1>best['f1']: best={'class':c,'threshold':round(float(t),3),'f1':round(float(f1),5),'precision':round(float(pr),5),'recall':round(float(rc),5)}
        thresholds.append(best)
    with open(root/'thresholds.json','w') as f: json.dump(thresholds,f,indent=2)
    local=[]; seen=Counter(); gi=0; limit=CFG.get('xai_local_examples',6); lime_limit=CFG.get('xai_lime_examples',2); per_class=max(1,limit//max(1,len(class_names)//2))
    for imgs,lbls in loader:
        for k in range(imgs.size(0)):
            if gi>=len(preds): break
            true=int(lbls[k]); pred=int(preds[gi]); p=pred if pred<len(class_names) else true
            if seen[true]<per_class:
                img=imgs[k:k+1].to(DEVICE); cam=grad_cam(model,img,p,c1m); drops=occlusion_drops(model,img,p,c1m,T,CFG.get('xai_occlusion_grid',4))[:8]
                im=denorm_img(imgs[k]); lime_heat,lime_rows=(None,None)
                if len(local)<lime_limit: lime_heat,lime_rows=lime_explain(model,img,p,c1m,T,CFG.get('xai_lime_segments',28),CFG.get('xai_lime_samples',64))
                cols=4 if lime_heat is not None else 3; fig,ax=plt.subplots(1,cols,figsize=(4*cols,4))
                ax[0].imshow(im); ax[0].set_title(f'true {class_names[true]}'); ax[0].axis('off')
                ax[1].imshow(im); ax[1].set_title(f'gradcam {class_names[p]}'); ax[1].axis('off')
                if cam is not None: ax[1].imshow(cam,cmap='jet',alpha=.45)
                pos=2
                if lime_heat is not None:
                    ax[2].imshow(im); ax[2].imshow(lime_heat,cmap='magma',alpha=.45); ax[2].set_title('lime segments'); ax[2].axis('off'); pos=3
                ax[pos].barh([str(d['cell']) for d in drops[::-1]],[d['drop'] for d in drops[::-1]]); ax[pos].set_title('occlusion drop')
                fig.tight_layout(); out=root/f'local_{len(local)}.png'; fig.savefig(out,dpi=160); plt.close(fig)
                local.append({'true':class_names[true],'predicted':class_names[p],'confidence':round(float(probs[gi,p]),5),'occlusion':drops,'lime':lime_rows,'plot':str(out)})
                seen[true]+=1
            gi+=1
            if len(local)>=limit: break
        if len(local)>=limit: break
    with open(root/'local_xai.json','w') as f: json.dump(local,f,indent=2)
    print(f'  XAI saved: {root}')
    return report

print('Eval and XAI ready')

Eval and XAI ready


## 9 · Generic Two-Phase Trainer

In [10]:
def train_model(model,train_dl,val_dl,test_dl,crit,save_path,class_names,
                c1m=None,mixup_alpha=None,cutmix_prob=None,epochs_frozen=None,epochs_full=None,patience=None,lr_frozen=None,lr_full=None,lr_backbone=None):
    ma=CFG['mixup_alpha'] if mixup_alpha is None else mixup_alpha
    cp=CFG['cutmix_prob'] if cutmix_prob is None else cutmix_prob
    ef=CFG['train_epochs_frozen'] if epochs_frozen is None else epochs_frozen
    eu=CFG['train_epochs_full'] if epochs_full is None else epochs_full
    pa=CFG['early_stop_patience'] if patience is None else patience
    lf=CFG['train_lr_frozen'] if lr_frozen is None else lr_frozen
    lu=CFG['train_lr_full'] if lr_full is None else lr_full
    lb=CFG['train_lr_backbone'] if lr_backbone is None else lr_backbone
    ema=EMA(model,CFG['ema_decay']); history=[]

    model.freeze_backbone()
    trainable=filter(lambda p:p.requires_grad,model.parameters())
    opt=torch.optim.AdamW(trainable,lr=lf,weight_decay=0.01)
    print(f'\nPhase 1: frozen backbone - {ef} epochs')
    for ep in range(ef):
        model.train(); freeze_batchnorm(model.backbone); tl,tc,tt=0,0,0
        for imgs,labels in train_dl:
            imgs,labels=imgs.to(DEVICE),labels.to(DEVICE)
            if np.random.random()<cp: imgs,ya,yb,lam=cutmix(imgs,labels)
            else: imgs,ya,yb,lam=mixup(imgs,labels,ma)
            if c1m:
                with torch.inference_mode(): e=c1m.embed(imgs)
                out=model(imgs,e)
            else: out=model(imgs)
            loss=mix_loss(crit,out,ya,yb,lam)
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); ema.update(model)
            tl+=loss.item()*imgs.size(0); tc+=(out.argmax(1)==labels).sum().item(); tt+=imgs.size(0)
        f1,_,_,_=evaluate(model,val_dl,c1m)
        row={'phase':'frozen','epoch':ep+1,'loss':round(tl/tt,6),'acc':round(tc/tt,6),'val_f1':round(float(f1),6),'lr':float(opt.param_groups[0]['lr'])}
        history.append(row); print(f'  Ep {ep+1}: loss={tl/tt:.4f} acc={tc/tt:.3f} val_f1={f1:.3f}')

    model.unfreeze_backbone()
    non_bb=[p for n,p in model.named_parameters() if 'backbone' not in n]
    opt=torch.optim.AdamW([{'params':model.backbone.parameters(),'lr':lb},{'params':non_bb,'lr':lu}],weight_decay=0.01)
    sched=torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt,T_0=len(train_dl)*max(3,min(5,eu)),T_mult=2)
    best,pat=-1,0
    print(f'\nPhase 2: full - {eu} epochs (patience={pa})')
    for ep in range(eu):
        model.train(); freeze_batchnorm(model.backbone); tl,tc,tt=0,0,0
        for imgs,labels in train_dl:
            imgs,labels=imgs.to(DEVICE),labels.to(DEVICE)
            if np.random.random()<cp: imgs,ya,yb,lam=cutmix(imgs,labels)
            else: imgs,ya,yb,lam=mixup(imgs,labels,ma)
            if c1m:
                with torch.inference_mode(): e=c1m.embed(imgs)
                out=model(imgs,e)
            else: out=model(imgs)
            loss=mix_loss(crit,out,ya,yb,lam)
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); sched.step(); ema.update(model)
            tl+=loss.item()*imgs.size(0); tc+=(out.argmax(1)==labels).sum().item(); tt+=imgs.size(0)
        f1,pf1,_,_=evaluate(model,val_dl,c1m); improved=f1>best; tag=''
        if improved:
            best=f1; pat=0; torch.save({'model_state':model.state_dict(),'ema':ema.shadow,'best_f1':best,'history':history,'class_names':class_names},save_path); tag=' *'
        else: pat+=1
        row={'phase':'full','epoch':ep+1,'loss':round(tl/tt,6),'acc':round(tc/tt,6),'val_f1':round(float(f1),6),'best':bool(improved),'lr_head':float(opt.param_groups[-1]['lr']),'lr_backbone':float(opt.param_groups[0]['lr'])}
        history.append(row); print(f'  Ep {ep+1}: loss={tl/tt:.4f} acc={tc/tt:.3f} val_f1={f1:.3f}{tag}')
        if pat>=pa: print(f'  Early stop at ep {ep+1}'); break

    ckpt=torch.load(save_path,map_location=DEVICE,weights_only=False); ckpt['history']=history; ckpt['class_names']=class_names; torch.save(ckpt,save_path)
    model.load_state_dict(ckpt['model_state'])
    f1_base,_,_,_=evaluate(model,val_dl,c1m)
    ema.apply(model); f1_ema,_,_,_=evaluate(model,val_dl,c1m)
    if f1_ema>=f1_base:
        print(f'  EMA: {f1_ema:.3f} >= base: {f1_base:.3f}')
        ckpt['model_state']=model.state_dict(); ckpt['ema_selected']=True; torch.save(ckpt,save_path)
    else:
        print(f'  Base: {f1_base:.3f} > EMA: {f1_ema:.3f}')
        model.load_state_dict(ckpt['model_state']); ckpt['ema_selected']=False; torch.save(ckpt,save_path)

    T=calibrate_T(model,val_dl,c1m,CFG.get('final_tta',False))
    decision=tune_decision(model,val_dl,class_names,c1m,T,CFG.get('final_tta',False))
    ckpt=torch.load(save_path,map_location=DEVICE,weights_only=False); ckpt['temperature']=T; ckpt['history']=history; ckpt['class_names']=class_names; ckpt['decision_config']=decision; ckpt['decision_val_f1']=decision.get('decision_val_f1'); torch.save(ckpt,save_path)
    save_learning_curve(history,save_path)

    tf1,tpf1,tp,tl_=evaluate(model,test_dl,c1m,T,CFG.get('final_tta',False),decision)
    print(f'\nTest macro F1: {tf1:.3f}')
    for c,f in zip(class_names,tpf1): print(f'  {c:22s} {f:.3f}')
    print(classification_report(tl_,tp,labels=list(range(len(class_names))),target_names=class_names,zero_division=0))
    save_xai_artifacts(model,test_dl,class_names,save_path,c1m,T,CFG.get('final_tta',False),decision)
    return best

print('Trainer ready')

Trainer ready


## 10 · Train CNN 1 — Content Type

In [11]:
if Path(CFG['cnn1_path']).exists():
    print('CNN 1 found - SKIPPING')
else:
    print('Scanning CNN1 datasets...')
    all_s=[]
    for name,fn in [('RVL-CDIP',scan_rvlcdip),('Tobacco',scan_tobacco),
                    ('ISIC',scan_isic),('MURA',scan_mura),('NIH',scan_nih),
                    ('Disaster',scan_disaster),('Buildings',scan_damaged),
                    ('Hurricane',scan_hurricane),('Screenshots',scan_screenshots),
                    ('Android',scan_android),('CEDAR',scan_cedar),
                    ('SigVer',scan_sigver),('SigNet',scan_signet),
                    ('AHCD',scan_ahcd),('Indoor',scan_indoor)]:
        try:
            s=fn()
            print(f'  {name:15s}: {len(s):>6}  | {dict(Counter(x[1] for x in s))}')
            all_s.extend(s)
        except Exception as e: print(f'  {name:15s}: FAIL ({e})')
    if not all_s: raise RuntimeError('No CNN1 data loaded')
    unknown={s[1] for s in all_s if s[1] not in C1_C2I}
    if unknown:
        print(f'  Dropping unknown labels: {unknown}')
        all_s=[s for s in all_s if s[1] in C1_C2I]
    print(f'\nRaw: {len(all_s)} | {dict(sorted(Counter(s[1] for s in all_s).items()))}')
    MAX_PER_CLASS=8000
    by_cls=defaultdict(list)
    for s in all_s: by_cls[s[1]].append(s)
    capped=[]
    for cls,samples in by_cls.items():
        if len(samples)>MAX_PER_CLASS:
            np.random.seed(42)
            idx=np.random.choice(len(samples),MAX_PER_CLASS,replace=False)
            samples=[samples[i] for i in idx]
        capped.extend(samples)
    all_s=capped
    print(f'Capped: {len(all_s)} | {dict(sorted(Counter(s[1] for s in all_s).items()))}')
    all_s=deduplicate(all_s)
    tr,va,te=prepare_splits(all_s)
    cw=compute_weights(tr,C1_C2I,power=CFG['cnn1_weight_power'],scale={'typed_document':1.10,'data_chart':0.96,'correspondence':1.02}).to(DEVICE)
    bs=CFG['train_batch_size']
    tr_dl=DataLoader(ImgDS(tr,C1_C2I,train_tf),bs,shuffle=True,
                     num_workers=2,pin_memory=True,drop_last=True)
    va_dl=DataLoader(ImgDS(va,C1_C2I,val_tf),bs,num_workers=2,pin_memory=True)
    te_dl=DataLoader(ImgDS(te,C1_C2I,val_tf),bs,num_workers=2,pin_memory=True)
    model=CNN1().to(DEVICE)
    crit=HierarchicalFocalLoss(gamma=CFG['focal_gamma'],weight=cw,ls=CFG['label_smoothing'],group_ids=C1_GROUP_ID,gw=CFG['cnn1_hierarchy_weight'])
    train_model(model,tr_dl,va_dl,te_dl,crit,CFG['cnn1_path'],CONTENT_CLASSES,
                mixup_alpha=CFG['cnn1_mixup_alpha'],cutmix_prob=CFG['cnn1_cutmix_prob'],epochs_full=CFG['cnn1_epochs_full'],patience=CFG['cnn1_patience'])
    del model,tr_dl,va_dl,te_dl; free_vram()

Scanning CNN1 datasets...
  RVL-CDIP       :  39997  | {'irrelevant': 5042, 'data_chart': 4994, 'correspondence': 7472, 'typed_document': 19957, 'handwritten': 2532}
  Tobacco        :   3482  | {'irrelevant': 230, 'correspondence': 1786, 'typed_document': 1265, 'handwritten': 201}
  ISIC           :  25331  | {'injury_photo': 25331}
  MURA           :  40009  | {'medical_record': 40009}
  NIH            :  19999  | {'medical_record': 19999}
  Disaster       :  13557  | {'scene_damage': 4320, 'irrelevant': 9237}
  Buildings      :  15240  | {'scene_damage': 15240}
  Hurricane      :  23000  | {'scene_damage': 15000, 'irrelevant': 8000}
  Screenshots    :    478  | {'screenshot': 478}
  Android        :  11999  | {'screenshot': 11999}
  CEDAR          :   2640  | {'signature_stamp': 2640}
  SigVer         :  14626  | {'signature_stamp': 14626}
  SigNet         :   2765  | {'signature_stamp': 2765}
  AHCD           :   3000  | {'handwritten': 3000}
  Indoor         :  15614  | {'irreleva

## 11 · Train CNN 3 — Image Quality

In [12]:
if Path(CFG['cnn3_path']).exists():
    print('CNN 3 found - SKIPPING')
else:
    print('Scanning CNN3 datasets...')
    all_q=[]
    for name,fn in [('KADID',scan_kadid),('BIQ2021',scan_biq2021)]:
        try:
            s=fn()
            print(f'  {name:15s}: {len(s):>6}  | {dict(Counter(x[1] for x in s))}')
            all_q.extend(s)
        except Exception as e: print(f'  {name:15s}: FAIL ({e})')
    if not all_q: print('WARNING: No quality data')
    else:
        all_q=[s for s in all_q if s[1] in C3_C2I]
        all_q=deduplicate(all_q)
        print(f'Total: {len(all_q)} | {dict(sorted(Counter(s[1] for s in all_q).items()))}')
        tr,va,te=prepare_splits(all_q)
        cw=compute_weights(tr,C3_C2I,power=CFG['cnn3_weight_power'],scale={'clear':1.04,'acceptable':0.98,'degraded':1.04}).to(DEVICE)
        bs=CFG['train_batch_size']
        tr_dl=DataLoader(ImgDS(tr,C3_C2I,train_tf_q3),bs,shuffle=True,
                         num_workers=2,pin_memory=True,drop_last=True)
        va_dl=DataLoader(ImgDS(va,C3_C2I,val_tf),bs,num_workers=2,pin_memory=True)
        te_dl=DataLoader(ImgDS(te,C3_C2I,val_tf),bs,num_workers=2,pin_memory=True)
        model=CNN3().to(DEVICE)
        crit=SoftOrdinalQualityLoss(weight=cw,ls=CFG['cnn3_label_smoothing'],ow=CFG['cnn3_ordinal_weight'],smooth=CFG['cnn3_soft_smoothing'])
        train_model(model,tr_dl,va_dl,te_dl,crit,CFG['cnn3_path'],QUALITY_CLASSES,
                    mixup_alpha=CFG['cnn3_mixup_alpha'],cutmix_prob=0.0,epochs_frozen=CFG['cnn3_epochs_frozen'],epochs_full=CFG['cnn3_epochs_full'],patience=CFG['cnn3_patience'],lr_full=CFG['cnn3_lr_full'],lr_backbone=CFG['cnn3_lr_backbone'])
        del model,tr_dl,va_dl,te_dl; free_vram()

Scanning CNN3 datasets...
  KADID          :  10125  | {'clear': 2743, 'acceptable': 3662, 'degraded': 3720}
  BIQ2021        :  12000  | {'degraded': 2926, 'clear': 2680, 'acceptable': 6394}
Total: 22125 | {'acceptable': 10056, 'clear': 5423, 'degraded': 6646}
  Split: 15487 train / 3319 val / 3319 test

Phase 1: frozen backbone - 3 epochs
  Ep 1: loss=1.0375 acc=0.505 val_f1=0.555
  Ep 2: loss=0.9626 acc=0.570 val_f1=0.563
  Ep 3: loss=0.9329 acc=0.592 val_f1=0.577

Phase 2: full - 24 epochs (patience=7)
  Ep 1: loss=0.8830 acc=0.637 val_f1=0.638 *
  Ep 2: loss=0.8459 acc=0.663 val_f1=0.655 *
  Ep 3: loss=0.8270 acc=0.674 val_f1=0.663 *
  Ep 4: loss=0.8201 acc=0.675 val_f1=0.668 *
  Ep 5: loss=0.8134 acc=0.683 val_f1=0.668
  Ep 6: loss=0.8107 acc=0.683 val_f1=0.671 *
  Ep 7: loss=0.7964 acc=0.693 val_f1=0.677 *
  Ep 8: loss=0.7819 acc=0.704 val_f1=0.682 *
  Ep 9: loss=0.7703 acc=0.714 val_f1=0.686 *
  Ep 10: loss=0.7632 acc=0.717 val_f1=0.685
  Ep 11: loss=0.7589 acc=0.723 val_f1=0.6

## 12 · Train CNN 4 — Tampering Detection

In [13]:
if Path(CFG['cnn4_path']).exists():
    print('CNN 4 found - SKIPPING')
else:
    print('Scanning CNN4 datasets...')
    all_t=[]
    for name,fn in [('CASIA',scan_casia),('RealFake',scan_realfake),('CG1050',scan_cg1050)]:
        try: s=fn(); print(f'  {name:15s}: {len(s):>6}'); all_t.extend(s)
        except Exception as e: print(f'  {name:15s}: FAIL ({e})')
    if not all_t: print('WARNING: No tamper data')
    else:
        MAX_T=15000
        by_cls=defaultdict(list)
        for s in all_t: by_cls[s[1]].append(s)
        capped=[]
        for cls,samples in by_cls.items():
            if len(samples)>MAX_T:
                np.random.seed(42)
                idx=np.random.choice(len(samples),MAX_T,replace=False)
                samples=[samples[i] for i in idx]
            capped.extend(samples)
        all_t=deduplicate(capped)
        print(f'Capped: {len(all_t)} | {dict(Counter(s[1] for s in all_t))}')
        tr,va,te=prepare_splits(all_t)
        cw=compute_weights(tr,TAMP_C2I).to(DEVICE)
        bs=CFG['train_batch_size']
        tr_dl=DataLoader(ImgDS(tr,TAMP_C2I,train_tf_c4),bs,shuffle=True,
                         num_workers=2,pin_memory=True,drop_last=True)
        va_dl=DataLoader(ImgDS(va,TAMP_C2I,val_tf),bs,num_workers=2,pin_memory=True)
        te_dl=DataLoader(ImgDS(te,TAMP_C2I,val_tf),bs,num_workers=2,pin_memory=True)
        model=CNN4().to(DEVICE)
        crit=nn.CrossEntropyLoss(weight=cw,label_smoothing=CFG['label_smoothing_cnn4'])
        train_model(model,tr_dl,va_dl,te_dl,crit,CFG['cnn4_path'],TAMPER_CLASSES,
                    mixup_alpha=0.0,cutmix_prob=0.0,epochs_full=CFG['cnn4_epochs_full'],patience=CFG['cnn4_patience'])
        ckpt=torch.load(CFG['cnn4_path'],map_location=DEVICE,weights_only=False)
        model.load_state_dict(ckpt['model_state']); model.eval()
        T4=ckpt.get('temperature',1.0); dec=ckpt.get('decision_config',{})
        lo,la=collect_logits(model,te_dl,None,CFG.get('final_tta',False))
        probs=decision_probs(lo,T4,dec).cpu().numpy(); labels_all=la.cpu().numpy()
        probs_all=probs[:,1]
        auc=roc_auc_score(labels_all,probs_all)
        thr=float(dec.get('threshold',CFG['tampering_threshold'])) if dec.get('type')=='binary_threshold' else CFG['tampering_threshold']
        preds=decision_preds(probs,dec)
        print(f'\nTest AUC: {auc:.3f} | F1: {f1_score(labels_all,preds,average="macro",zero_division=0):.3f} | threshold={thr:.3f}')
        del model,tr_dl,va_dl,te_dl; free_vram()

Scanning CNN4 datasets...
  CASIA          :  12614
  RealFake       :  57589
  CG1050         :   2088
  Dedup: 30000 -> 29776
Capped: 29776 | {'authentic': 14776, 'tampered': 15000}
  Split: 20842 train / 4467 val / 4467 test

Phase 1: frozen backbone - 4 epochs
  Ep 1: loss=0.5644 acc=0.718 val_f1=0.709
  Ep 2: loss=0.5341 acc=0.742 val_f1=0.744
  Ep 3: loss=0.5217 acc=0.752 val_f1=0.753
  Ep 4: loss=0.5149 acc=0.756 val_f1=0.758

Phase 2: full - 14 epochs (patience=5)
  Ep 1: loss=0.3794 acc=0.851 val_f1=0.869 *
  Ep 2: loss=0.2859 acc=0.904 val_f1=0.896 *
  Ep 3: loss=0.2567 acc=0.920 val_f1=0.902 *
  Ep 4: loss=0.2393 acc=0.930 val_f1=0.907 *
  Ep 5: loss=0.2335 acc=0.932 val_f1=0.908 *
  Ep 6: loss=0.2410 acc=0.928 val_f1=0.915 *
  Ep 7: loss=0.2221 acc=0.938 val_f1=0.916 *
  Ep 8: loss=0.2080 acc=0.945 val_f1=0.922 *
  Ep 9: loss=0.1990 acc=0.952 val_f1=0.923 *
  Ep 10: loss=0.1887 acc=0.955 val_f1=0.921
  Ep 11: loss=0.1822 acc=0.960 val_f1=0.923
  Ep 12: loss=0.1731 acc=0.965

## 13 · Train CNN 2 — Evidentiary Weight

In [14]:
if Path(CFG['cnn2_path']).exists():
    print('CNN 2 found - SKIPPING')
else:
    print('Building CNN2 proxy labels...')
    c1=CNN1().to(DEVICE)
    ckpt1=torch.load(CFG['cnn1_path'],map_location=DEVICE,weights_only=False)
    c1.load_state_dict(ckpt1['model_state']); c1.eval()
    T1=ckpt1.get('temperature',1.0)
    embed_dim=c1.embed(torch.randn(1,3,sz,sz).to(DEVICE)).shape[1]

    c3=CNN3().to(DEVICE)
    ckpt3=torch.load(CFG['cnn3_path'],map_location=DEVICE,weights_only=False)
    c3.load_state_dict(ckpt3['model_state']); c3.eval()
    T3=ckpt3.get('temperature',1.0)

    src_samples=[]
    for fn in [scan_rvlcdip,scan_tobacco,scan_isic,scan_mura,scan_nih,
               scan_disaster,scan_damaged,scan_hurricane,scan_screenshots,
               scan_android,scan_cedar,scan_sigver,scan_signet,scan_ahcd,scan_indoor]:
        try: s=fn(); src_samples.extend(s[:3000])
        except: pass
    print(f'  Source pool: {len(src_samples)}')

    CONTENT_WEIGHT={
        'injury_photo':1.00,'medical_record':1.00,'typed_document':0.85,
        'signature_stamp':0.80,'correspondence':0.70,'scene_damage':0.65,
        'handwritten':0.55,'data_chart':0.50,'screenshot':0.30,'irrelevant':0.00,
    }
    QUALITY_MULT={'clear':1.00,'acceptable':0.75,'degraded':0.40}

    scores=[]; valid_paths=[]
    for path,_,src in src_samples:
        try:
            img=Image.open(path).convert('RGB')
            it=val_tf(img).unsqueeze(0).to(DEVICE)
            with torch.inference_mode():
                c1p=F.softmax(c1(it)/T1,dim=1).squeeze().cpu().numpy()
                c3p=F.softmax(c3(it)/T3,dim=1).squeeze().cpu().numpy()
            cs=sum(CONTENT_WEIGHT.get(C1_I2C[i],0)*float(c1p[i]) for i in range(N_C1))
            qm=sum(QUALITY_MULT.get(C3_I2C[i],0)*float(c3p[i]) for i in range(N_C3))
            scores.append(cs*qm); valid_paths.append((path,src))
        except: pass

    thr=float(np.percentile(scores,60))
    print(f'  Scores [{min(scores):.3f},{max(scores):.3f}] | p60 threshold: {thr:.3f}')
    proxy=[(p,'primary_evidence' if sc>=thr else 'secondary_evidence',src)
           for (p,src),sc in zip(valid_paths,scores)]
    print(f'  Proxy: {len(proxy)} | {dict(Counter(x[1] for x in proxy))}')
    del c3; free_vram()

    if len(proxy)<100: print('Not enough proxy samples')
    else:
        proxy=deduplicate(proxy)
        tr,va,te=prepare_splits(proxy)
        cw=compute_weights(tr,C2_C2I).to(DEVICE)
        bs=CFG['train_batch_size_c2']
        tr_dl=DataLoader(ImgDS(tr,C2_C2I,train_tf),bs,shuffle=True,
                         num_workers=2,pin_memory=True,drop_last=True)
        va_dl=DataLoader(ImgDS(va,C2_C2I,val_tf),bs,num_workers=2,pin_memory=True)
        te_dl=DataLoader(ImgDS(te,C2_C2I,val_tf),bs,num_workers=2,pin_memory=True)
        model=CNN2(embed_dim=embed_dim).to(DEVICE)
        crit=nn.CrossEntropyLoss(weight=cw,label_smoothing=CFG['label_smoothing'])
        train_model(model,tr_dl,va_dl,te_dl,crit,CFG['cnn2_path'],EVIDENCE_CLASSES,
                    c1m=c1,mixup_alpha=CFG['mixup_alpha'],cutmix_prob=CFG['cutmix_prob'],epochs_full=CFG['cnn2_epochs_full'],patience=CFG['cnn2_patience'])
        del model,tr_dl,va_dl,te_dl; free_vram()
    del c1; free_vram()

Building CNN2 proxy labels...
  Source pool: 41883
  Scores [0.001,0.993] | p60 threshold: 0.399
  Proxy: 41881 | {'secondary_evidence': 25128, 'primary_evidence': 16753}
  Dedup: 41881 -> 41482
  Split: 29036 train / 6223 val / 6223 test

Phase 1: frozen backbone - 4 epochs
  Ep 1: loss=0.4494 acc=0.749 val_f1=0.933
  Ep 2: loss=0.4401 acc=0.751 val_f1=0.930
  Ep 3: loss=0.4339 acc=0.746 val_f1=0.937
  Ep 4: loss=0.4315 acc=0.755 val_f1=0.935

Phase 2: full - 10 epochs (patience=4)
  Ep 1: loss=0.4184 acc=0.756 val_f1=0.940 *
  Ep 2: loss=0.4123 acc=0.776 val_f1=0.919
  Ep 3: loss=0.4096 acc=0.758 val_f1=0.947 *
  Ep 4: loss=0.3947 acc=0.768 val_f1=0.948 *
  Ep 5: loss=0.3995 acc=0.775 val_f1=0.949 *
  Ep 6: loss=0.4097 acc=0.760 val_f1=0.943
  Ep 7: loss=0.4041 acc=0.758 val_f1=0.947
  Ep 8: loss=0.4035 acc=0.760 val_f1=0.945
  Ep 9: loss=0.3975 acc=0.770 val_f1=0.949
  Early stop at ep 9
  Base: 0.949 > EMA: 0.948
  T=0.545
  Decision=binary_threshold val_f1 0.949523 -> 0.949861

Te

## 14 - Training Report


In [15]:
def model_bytes(path):
    return round(Path(path).stat().st_size/1024/1024,3) if Path(path).exists() else None

def model_params(model):
    return int(sum(p.numel() for p in model.parameters()))

def read_metrics(path):
    mp=Path(CFG['output_dir'])/'xai'/Path(path).stem/'metrics.json'
    if mp.exists():
        with open(mp) as f: return json.load(f)
    return {}

def checkpoint_meta(path):
    if not Path(path).exists(): return {}
    ck=torch.load(path,map_location='cpu',weights_only=False)
    dec=ck.get('decision_config',{'type':'argmax'})
    return {'best_val_f1':round(float(ck.get('best_f1',0)),5),'decision_val_f1':ck.get('decision_val_f1'),'temperature':round(float(ck.get('temperature',1.0)),5),'ema_selected':bool(ck.get('ema_selected',False)),'decision_config':dec}

def latency_ms(name,path,reps=None):
    if not Path(path).exists(): return None
    reps=CFG.get('report_latency_reps',12) if reps is None else reps
    try:
        dummy=torch.randn(1,3,sz,sz).to(DEVICE)
        c1m=None
        if name=='cnn1_content': m=CNN1().to(DEVICE)
        elif name=='cnn2_evidence':
            c1m=CNN1().to(DEVICE); ck1=torch.load(CFG['cnn1_path'],map_location=DEVICE,weights_only=False); c1m.load_state_dict(ck1['model_state']); c1m.eval()
            ck=torch.load(path,map_location=DEVICE,weights_only=False); ed=ck['model_state']['head.1.weight'].shape[1]-2048; m=CNN2(embed_dim=ed).to(DEVICE)
        elif name=='cnn3_quality': m=CNN3().to(DEVICE)
        elif name=='cnn4_tamper': m=CNN4().to(DEVICE)
        else: return None
        ck=torch.load(path,map_location=DEVICE,weights_only=False); m.load_state_dict(ck['model_state']); m.eval()
        with torch.inference_mode():
            for _ in range(3): forward_logits(m,dummy,c1m)
            if torch.cuda.is_available(): torch.cuda.synchronize()
            t0=time.time()
            for _ in range(reps): forward_logits(m,dummy,c1m)
            if torch.cuda.is_available(): torch.cuda.synchronize()
        out=round((time.time()-t0)*1000/reps,3)
        del m
        if c1m is not None: del c1m
        free_vram(); return out
    except Exception as e:
        free_vram(); return None

def write_training_report():
    specs=[('cnn1_content',CFG['cnn1_path'],CONTENT_CLASSES,CNN1),('cnn2_evidence',CFG['cnn2_path'],EVIDENCE_CLASSES,None),('cnn3_quality',CFG['cnn3_path'],QUALITY_CLASSES,CNN3),('cnn4_tamper',CFG['cnn4_path'],TAMPER_CLASSES,CNN4)]
    rows=[]
    for name,path,classes,builder in specs:
        meta=checkpoint_meta(path); met=read_metrics(path); params=None
        if builder is not None:
            try:
                m=builder(); params=model_params(m); del m
            except Exception: params=None
        elif Path(path).exists():
            try:
                ck=torch.load(path,map_location='cpu',weights_only=False); ed=ck['model_state']['head.1.weight'].shape[1]-2048; m=CNN2(embed_dim=ed); params=model_params(m); del m
            except Exception: params=None
        row={'model':name,'classes':len(classes),'labels':classes,'params':params,'model_mb':model_bytes(path),'latency_ms':latency_ms(name,path),'best_val_f1':meta.get('best_val_f1'),'test_macro_f1':met.get('macro_f1'),'accuracy':met.get('accuracy'),'auc':met.get('auc'),'ece':met.get('ece'),'temperature':meta.get('temperature'),'ema_selected':meta.get('ema_selected'),'decision_val_f1':meta.get('decision_val_f1'),'decision_config':meta.get('decision_config'),'path':path}
        rows.append(row)
    out=Path(CFG['output_dir']); out.mkdir(parents=True,exist_ok=True)
    df=pd.DataFrame(rows); df.to_csv(out/'model_comparison.csv',index=False)
    with open(out/'model_comparison.json','w') as f: json.dump(rows,f,indent=2)
    manifest={'pipeline':'document_processing_pretrained_cv','classes':{'cnn1_content':CONTENT_CLASSES,'cnn2_evidence':EVIDENCE_CLASSES,'cnn3_quality':QUALITY_CLASSES,'cnn4_tamper':TAMPER_CLASSES},'config':{k:v for k,v in CFG.items() if isinstance(v,(str,int,float,bool))},'models':rows,'xai_outputs':{r['model']:str(Path(CFG['output_dir'])/'xai'/Path(r['path']).stem) for r in rows},'integration_outputs':['*_pipeline.json','model_comparison.json','run_manifest.json','integration_schema.json']}
    with open(out/'run_manifest.json','w') as f: json.dump(manifest,f,indent=2)
    schema={'input':'pdf/docx/pptx/image document','stages':['normalise','layout_regions','content_classification','evidence_weighting','quality_check','tamper_check','ocr','captioning','json_assembly'],'element_schema':{'id':'region id','page':'page index','bbox':'page coordinates','content_type':CONTENT_CLASSES,'content_conf':'softmax confidence','evid_weight':EVIDENCE_CLASSES,'quality':QUALITY_CLASSES,'tamper_label':['likely_authentic','needs_verification','possibly_tampered'],'tamper_threshold':'checkpoint decision threshold when available','quality_usability':['usable','low_quality'],'text':'ocr text when present','caption':'vision caption when present'},'rag_payload':{'document_json':'pipeline output','retrieved_legal_context':'local vector database results','llm_task':'legal feedback and next action recommendations'}}
    with open(out/'integration_schema.json','w') as f: json.dump(schema,f,indent=2)
    lines=['# Model Card','','Pretrained transfer-learning document pipeline for extracting structured context from legal or evidence documents.','','## Models']
    for r in rows:
        lines.append(f"- {r['model']}: labels={', '.join(r['labels'])}; test_macro_f1={r.get('test_macro_f1')}; ece={r.get('ece')}; auc={r.get('auc')}; decision={r.get('decision_config',{}).get('type')}")
    lines += ['','## XAI','Grad-CAM, perturbation occlusion, LIME-style segment perturbation, calibration curves, thresholds, confusion matrices, and local examples are saved under results/xai.','','## Integration','The exported JSON is intended for backend/RAG/LLM use. The CV models extract evidence structure; the legal RAG system should retrieve law/context and the LLM should generate feedback from the retrieved context plus this JSON.','','## Limits','Quality labels are subjective, proxy evidence labels are model-derived, and legal recommendations require RAG grounding and human review.']
    with open(out/'model_card.md','w',encoding='utf-8') as f: f.write('\n'.join(lines))
    print(df[['model','test_macro_f1','accuracy','auc','ece','latency_ms','model_mb']])
    return rows

TRAINING_REPORT=write_training_report()
print('Training report ready')

           model  test_macro_f1  accuracy      auc      ece  latency_ms  \
0   cnn1_content        0.93751   0.94268  0.99633  0.00693      16.508   
1  cnn2_evidence        0.94814   0.94986  0.99042  0.00782         NaN   
2   cnn3_quality        0.72497   0.72431  0.87338  0.01909       8.014   
3    cnn4_tamper        0.93843   0.93844  0.98382  0.01016      10.874   

   model_mb  
0   160.708  
1   189.079  
2    32.401  
3    51.766  
Training report ready


## 14 · Pretrained Ensemble Boosters

In [16]:
class QualityBooster:
    def __init__(self):
        try:
            import pyiqa
            self.m=pyiqa.create_metric('clipiqa',device=DEVICE); self.ok=True
            print('  CLIP-IQA loaded')
        except Exception as e: print(f'  CLIP-IQA failed: {e}'); self.ok=False
    @torch.no_grad()
    def score(self,t):
        if not self.ok: return None
        return self.m(t).squeeze().cpu().numpy()
    def unload(self):
        if self.ok: del self.m
        free_vram()

class TamperBooster:
    def __init__(self):
        try:
            from transformers import pipeline as hf_pipe
            self.pipe=hf_pipe('image-classification',
                model='dima806/deepfake_vs_real_image_detection',
                device=0 if torch.cuda.is_available() else -1)
            self.ok=True; print('  ViT forgery detector loaded')
        except Exception as e: print(f'  ViT failed: {e}'); self.ok=False
    def predict(self,pil):
        if not self.ok: return 0.5
        try:
            r=self.pipe(pil)
            for x in r:
                if 'fake' in x['label'].lower(): return x['score']
            return 1.0-r[0]['score']
        except: return 0.5
    def unload(self):
        if self.ok: del self.pipe
        free_vram()

print('Boosters defined')

Boosters defined


## 15 · Stage 1 — Layout Detection

In [17]:
class LayoutDetector:
    def __init__(self):
        try:
            from surya.detection import DetectionModel
            from surya.layout import LayoutModel
            self.dm=DetectionModel(); self.lm=LayoutModel()
            self.ok=True; print('  Surya loaded')
        except Exception as e:
            print(f'  Surya fallback ({e})'); self.ok=False
    def detect(self,page):
        if not self.ok:
            w,h=page.size
            return [{'bbox':[0,0,w,h],'type':'figure','conf':1.0}]
        try:
            from surya.detection import detect
            from surya.layout import layout
            dr=detect([page],self.dm); lr=layout([page],self.lm,dr)
            regs=[{'bbox':b.bbox,
                   'type':getattr(b,'label','figure').lower(),
                   'conf':getattr(b,'confidence',0.9)}
                  for b in lr[0].bboxes]
            return regs if regs else [{'bbox':[0,0,page.size[0],page.size[1]],
                                       'type':'figure','conf':1.0}]
        except:
            w,h=page.size
            return [{'bbox':[0,0,w,h],'type':'figure','conf':1.0}]
    def unload(self):
        if self.ok: del self.dm,self.lm
        free_vram()

print('Layout ready')

Layout ready


## 16 · Captioning

In [18]:
PROMPTS={
    'injury_photo':   'Describe visible physical details: location, size, colour, appearance. No cause or severity.',
    'scene_damage':   'Describe condition of objects and surfaces. What appears damaged or displaced. No cause.',
    'screenshot':     'Transcribe all visible text exactly. Note platform, timestamps, sender names.',
    'typed_document': 'Describe layout, structure, visible dates, reference numbers, stamps, signatures.',
    'correspondence': 'Describe layout, sender/recipient, dates, subject matter, tone and key assertions.',
    'handwritten':    'Describe writing style and transcribe all legible portions.',
    'medical_record': 'Describe medical information, form fields, clinical measurements. No interpretation.',
    'data_chart':     'Describe chart type, axis labels, data ranges, visible trends.',
    'signature_stamp':'Describe style, readable text, hand-drawn versus printed.',
    'unclassifiable': 'Describe all visible objects, text, and spatial layout objectively.',
}

class Captioner:
    def __init__(self):
        try:
            from transformers import AutoModelForCausalLM,AutoTokenizer
            dt=torch.float16 if torch.cuda.is_available() else torch.float32
            self.model=AutoModelForCausalLM.from_pretrained(
                'vikhyatk/moondream2',trust_remote_code=True,
                torch_dtype=dt).to(DEVICE).eval()
            self.tok=AutoTokenizer.from_pretrained(
                'vikhyatk/moondream2',trust_remote_code=True)
            self.ok=True; print('  moondream2 loaded')
        except Exception as e:
            print(f'  moondream2 failed ({e})'); self.ok=False
    def caption(self,pil,ct='unclassifiable'):
        if not self.ok: return 'Captioning unavailable'
        try:
            enc=self.model.encode_image(pil)
            return self.model.answer_question(
                enc,PROMPTS.get(ct,PROMPTS['unclassifiable']),self.tok)
        except: return 'Caption failed'
    def unload(self):
        if self.ok: del self.model,self.tok
        free_vram()

print('Captioner ready')

Captioner ready


## 17 · Text Extraction

In [19]:
class OCR:
    def __init__(self): self.p=None
    def _init(self):
        if self.p is None:
            try:
                from paddleocr import PaddleOCR
                self.p=PaddleOCR(use_angle_cls=True,lang='en',show_log=False)
            except: pass
    def extract(self,pil):
        self._init()
        if self.p is None: return ''
        try:
            r=self.p.ocr(np.array(pil),cls=True)
            if r and r[0]:
                return ' '.join(l[1][0] for l in r[0] if l[1][1]>0.5)
        except: pass
        return ''
    def unload(self): del self.p; self.p=None; free_vram()

print('OCR ready')

OCR ready


## 18 · Colour & Similarity

In [20]:
def dominant_colours(pil,n=3):
    img=np.array(pil.resize((64,64))).reshape(-1,3)
    km=KM(n_clusters=n,n_init=1,random_state=0).fit(img)
    return [list(map(int,c)) for c in km.cluster_centers_]

def cos_sim(a,b):
    a,b=np.array(a),np.array(b)
    return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-8))

print('Utils ready')

Utils ready


## 19 · JSON Assembly

In [21]:
def assemble(elements,meta,native_text=None):
    out={'file':meta.get('filename','-'),
         'file_hash':meta.get('hash',''),
         'pages':meta.get('n_pages',0),
         'elements':[],'native_text':native_text or {}}
    embs={e['id']:e.get('embedding',[]) for e in elements if e.get('embedding')}
    for e in elements:
        if e.get('embedding'):
            e['similar']=[{'id':eid,'sim':round(cos_sim(e['embedding'],emb),3)}
                          for eid,emb in embs.items()
                          if eid!=e['id'] and cos_sim(e['embedding'],emb)>0.85]
        e.pop('embedding',None); out['elements'].append(e)
    js=json.dumps(out); out['est_tokens']=len(js)//4
    if out['est_tokens']>200000: out['token_warning']='exceeds_200k'
    elif out['est_tokens']>100000: out['token_warning']='exceeds_100k'
    return out

print('Assembly ready')

Assembly ready


## 20 · Pipeline Orchestrator

In [22]:

def predict_tta(model,it,T,c1m=None,decision=None):
    with torch.inference_mode():
        out=forward_logits(model,it,c1m)
        if CFG.get('inference_tta',False): out=(out+forward_logits(model,torch.flip(it,dims=[3]),c1m))/2
        return decision_probs(out,T,decision).squeeze().cpu().numpy()

def choose_idx(p,decision=None):
    if decision and decision.get('type')=='binary_threshold' and len(p)==2:
        return int(float(p[1])>=float(decision.get('threshold',0.5)))
    return int(np.argmax(p))

def binary_threshold(decision,default=0.5):
    return float(decision.get('threshold',default)) if decision and decision.get('type')=='binary_threshold' else default

def risk_band(prob,thr):
    lo=max(0.05,thr-0.15); hi=min(0.95,thr+0.15)
    if prob<lo: return 'likely_authentic'
    if prob<hi: return 'needs_verification'
    return 'possibly_tampered'

def process_document(filepath):
    print(f'\n{"="*70}\n{Path(filepath).name}\n{"="*70}')
    t0=time.time(); fp=Path(filepath)
    meta={'filename':fp.name,'hash':sha256_file(fp)}

    pages=normalize_file(filepath); native=extract_native_text(filepath)
    meta['n_pages']=len(pages)
    print(f'  S0: {len(pages)} pages, {len(native)} native text blocks')
    if not pages and not native: print('  Nothing to process'); return None

    ld=LayoutDetector(); regions=[]
    for pi,pg in enumerate(pages):
        for ri,reg in enumerate(ld.detect(pg)):
            regions.append({'pg':pi,'ri':ri,'bbox':reg['bbox'],
                            'type':reg['type'],'img':pg})
    ld.unload()
    for r in regions:
        x0,y0,x1,y1=[int(c) for c in r['bbox']]
        r['crop']=r['img'].crop((x0,y0,x1,y1))
    print(f'  S1: {len(regions)} regions')

    ck1=torch.load(CFG['cnn1_path'],map_location=DEVICE,weights_only=False)
    m1=CNN1().to(DEVICE); m1.load_state_dict(ck1['model_state']); m1.eval()
    T1=ck1.get('temperature',1.0); dec1=ck1.get('decision_config',{'type':'argmax'})
    els=[]
    for r in regions:
        el={'id':f'r{r["pg"]}_{r["ri"]}','type':'image_region',
            'page':r['pg'],'bbox':r['bbox']}
        it=val_tf(r['crop']).unsqueeze(0).to(DEVICE)
        with torch.inference_mode():
            p=predict_tta(m1,it,T1,decision=dec1)
            emb=m1.embed(it).squeeze().cpu().numpy()
        ti=choose_idx(p,dec1)
        el['content_type']=C1_I2C[ti]
        el['content_conf']=round(float(p[ti]),4)
        el['content_scores']={C1_I2C[i]:round(float(p[i]),4) for i in range(N_C1)}
        order=np.argsort(p); el['runner_up']=C1_I2C[order[-2]]
        el['content_margin']=round(float(p[order[-1]]-p[order[-2]]),4)
        text_like={'typed_document','correspondence','data_chart','handwritten','signature_stamp'}
        el['ambiguous_content']=bool(el['content_type'] in text_like and el['runner_up'] in text_like and el['content_margin']<CFG['ambiguity_gap'])
        el['unclassifiable']=float(p[ti])<CFG['ood_max_softmax']
        el['embedding']=emb.tolist(); els.append(el)

    if Path(CFG['cnn2_path']).exists():
        ck2=torch.load(CFG['cnn2_path'],map_location=DEVICE,weights_only=False); dec2=ck2.get('decision_config',{'type':'argmax'})
        embed_dim=ck2['model_state']['head.1.weight'].shape[1]-2048
        m2=CNN2(embed_dim=embed_dim).to(DEVICE); m2.load_state_dict(ck2['model_state']); m2.eval()
        for el,r in zip(els,regions):
            it=val_tf(r['crop']).unsqueeze(0).to(DEVICE)
            with torch.inference_mode():
                p=predict_tta(m2,it,ck2.get('temperature',1.0),m1,dec2)
            ei=choose_idx(p,dec2); el['evid_weight']=C2_I2C[ei]
            el['evid_scores']={C2_I2C[i]:round(float(p[i]),4) for i in range(N_C2)}
        del m2; free_vram()
    del m1; free_vram()

    if Path(CFG['cnn3_path']).exists():
        ck3=torch.load(CFG['cnn3_path'],map_location=DEVICE,weights_only=False)
        m3=CNN3().to(DEVICE); m3.load_state_dict(ck3['model_state']); m3.eval()
        T3=ck3.get('temperature',1.0); dec3=ck3.get('decision_config',{'type':'argmax'})
        for el,r in zip(els,regions):
            it=val_tf(r['crop']).unsqueeze(0).to(DEVICE)
            with torch.inference_mode():
                p=predict_tta(m3,it,T3,decision=dec3)
            qi=choose_idx(p,dec3); el['quality']=C3_I2C[qi]
            el['quality_usability']='low_quality' if el['quality']=='degraded' else 'usable'
            el['quality_scores']={C3_I2C[i]:round(float(p[i]),4) for i in range(N_C3)}
        del m3; free_vram()

    if Path(CFG['cnn4_path']).exists():
        ck4=torch.load(CFG['cnn4_path'],map_location=DEVICE,weights_only=False)
        m4=CNN4().to(DEVICE); m4.load_state_dict(ck4['model_state']); m4.eval()
        T4=ck4.get('temperature',1.0); dec4=ck4.get('decision_config',{'type':'argmax'}); thr4=binary_threshold(dec4,CFG['tampering_threshold'])
        for el,r in zip(els,regions):
            if el.get('content_type')=='irrelevant': continue
            it=val_tf(r['crop']).unsqueeze(0).to(DEVICE)
            with torch.inference_mode():
                p=predict_tta(m4,it,T4,decision=dec4)
            el['tamper_prob']=round(float(p[1]),4)
            el['tamper_threshold']=round(float(thr4),4)
            el['tamper_label']=risk_band(float(p[1]),thr4)
        del m4; free_vram()

    tb=TamperBooster()
    for el,r in zip(els,regions):
        if el.get('content_type')!='irrelevant':
            vit_p=tb.predict(r['crop']); el['tamper_vit']=round(vit_p,4)
            if 'tamper_prob' in el:
                ens=0.55*el['tamper_prob']+0.45*vit_p
                el['tamper_ensemble']=round(ens,4); el['tamper_ensemble_label']=risk_band(float(ens),el.get('tamper_threshold',CFG['tampering_threshold']))
    tb.unload()

    if CFG['run_captioning']:
        cap=Captioner()
        for el,r in zip(els,regions):
            ct=el.get('content_type','unclassifiable')
            if ct=='irrelevant': continue
            if el.get('unclassifiable'): ct='unclassifiable'
            el['caption']=cap.caption(r['crop'],ct)
        cap.unload()

    if CFG['run_ocr']:
        ocr=OCR()
        for el,r in zip(els,regions):
            if r['type'] in ('text','heading','list','caption','footnote'):
                el['text']=ocr.extract(r['crop'])
        ocr.unload()

    for el,r in zip(els,regions):
        if el.get('content_type')!='irrelevant':
            el['colours']=dominant_colours(r['crop'])

    for r in regions: r.pop('img',None); r.pop('crop',None)

    out=assemble(els,meta,native)
    out['time_s']=round(time.time()-t0,2)
    op=Path(CFG['output_dir'])/f'{fp.stem}_pipeline.json'
    Path(CFG['output_dir']).mkdir(parents=True,exist_ok=True)
    with open(op,'w') as f: json.dump(out,f,indent=2,default=str)
    print(f'  Done: {op.name} ({op.stat().st_size/1024:.1f} KB) in {out["time_s"]}s')
    return out

print('Pipeline ready')

Pipeline ready


## 21 · Run Pipeline

In [23]:
inp=Path(CFG['input_dir'])
files=[]
if inp.exists():
    for ext in ('*.jpg','*.jpeg','*.png','*.pdf','*.docx','*.doc','*.pptx'):
        files.extend(inp.rglob(ext))
if not files:
    print(f'No files in {inp}')
else:
    print(f'{len(files)} files found')
    for fp in files[:5]:
        try: process_document(str(fp))
        except Exception as e: print(f'  FAILED: {e}')

No files in /kaggle/input/test-documents


## 22 · Inspect Results

In [24]:
jsons=sorted(glob.glob(os.path.join(CFG['output_dir'],'*_pipeline.json')))
if jsons:
    with open(jsons[0]) as f: s=json.load(f)
    print(f'File: {s["file"]} | Pages: {s["pages"]} | '
          f'Elements: {len(s["elements"])} | Tokens: {s.get("est_tokens","-")}')
    if s['elements']:
        e=s['elements'][0]; print('\nFirst element:')
        for k,v in e.items(): print(f'  {k}: {v}')
else: print('No results yet')

No results yet


## 23 · Download Models

In [25]:
import shutil
os.makedirs("/kaggle/working/output", exist_ok=True)
for f in ["cnn1_content.pt", "cnn2_evidence.pt", "cnn3_quality.pt", "cnn4_tamper.pt"]:
    src=f"/kaggle/working/{f}"
    if os.path.exists(src):
        shutil.copy(src, f"/kaggle/working/output/{f}")
        print(f'Copied {f}')

Copied cnn1_content.pt
Copied cnn2_evidence.pt
Copied cnn3_quality.pt
Copied cnn4_tamper.pt
